# C10 prereg44：競賽二 等級 CNN（CNN44）長訓練（Colab）

**不用改任何程式**：在「變更執行階段類型」選 GPU（G4；拿不到時 A100 也可），按「執行階段 → 全部執行」。

0. 把 `我的雲端硬碟/C10/p44/kit_parts/` 的分割檔接回 `C10_p44_kit.zip`（雜湊核對）。
1. 掛上雲端硬碟、核對 GPU、解開套件並逐檔核對。
2. 第一次先 bench 300 步，再訓練 200,000 步（G4 約 12–13 小時；每 20,000 步存到 `C10/p44/run/ckpt/`；斷線後重按「全部執行」會從最後的存檔接著跑）。
3. 8 個候選版本（第 140k／160k／180k／200k 步 × 原始／EMA）算留出題棋和 inner 棋的每盤 logits，存到 `C10/p44/logits/`。
4. 打包 `C10/p44/C10_p44_results.zip`，下載回傳。

任何一格出現紅字或 `STOP`，停下來，截最後 20 行。

In [ ]:
# 0. 把 Drive 上 C10/p44/kit_parts/ 的 38 份接回 C10_p44_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p44'
KIT = os.path.join(D, 'C10_p44_kit.zip')
SHA = 'ff20262c3a4a2ce3e50293958b49f8fe3dcb5567d01ea38f3098efd5230885da'
PART_SHA = {"kit.part00": "37ef59ffc515892febdc9338e3acf4f728b4ecb25338547307ddc85bc2ae0a0f", "kit.part01": "837e3c6539254c0111ebb5322235394325be7d5e978329fbd4d92c628a22e728", "kit.part02": "0151704f5454d8442c78836a39f1a9ca8333f8068ef51561add44ae02e00bb6a", "kit.part03": "4603d52334cc3721d3b8e6e935bd059779d47b0665106fa9adf24188f9b1fff5", "kit.part04": "9f0667e87db66e082514cb7a7669875541793192a441a7e8542b45d817c6753b", "kit.part05": "91bd4b71f43c9251e40265e69192b8524085a27291231730ea895c4407335e45", "kit.part06": "a567bb50b6a72edf77e52f11e0d6a7a651babd209cb477bd2a4ad570d7910300", "kit.part07": "4c490250c0b3ce2d9c27a9c37d8af43b97e43e4690e74e4e134beabaaf29d14f", "kit.part08": "2f1f1bde8f79d5a4b444b98b5fb00ffdd696d8e98036c0735ef92eb3375dd78c", "kit.part09": "deda2ac6313503736fbb4fa6506cbf16276e8ec9b8bbe05ed37786803731f67b", "kit.part10": "76f2f667c60bcb64afc8fd63d90ba45cf7833edceade1b1dc5f539636e5675c7", "kit.part11": "aece1211c303a6be3ed5eeb1c78f51be6637febd76957582d5a17fcad212dd77", "kit.part12": "613143717fdb9819ec8d5efdce7bf12c83d53c688d40cc44a1ebad8224535669", "kit.part13": "5407c1005ef154143448986f76d89994dff63ccb2c810397786a7022039fc458", "kit.part14": "160a82ec7277087b09424a5cd91891e352e8e7bd20613d507b41e66bf81ac098", "kit.part15": "800f458ae04e89801920b1e080cfb61a0853ee3a3a9090505966d619cd597e84", "kit.part16": "70e2d811c9cb18b7504aa4c6a5c994f66f40d86c33e72b0343ceae5b36898e4d", "kit.part17": "55e2c8ba328fde5f637ec0241bbfd273aece61ec33847b1d29057a0960383b81", "kit.part18": "973d9e31802d995bcfe9031cf26fc7956db359a5939ebf83fd3e9247fa1da7bd", "kit.part19": "f1761e24b56c151c74c20a3ba19e19d91a7435841c76e167b9cdfdcbd408dc1e", "kit.part20": "d45ced6017221988a29283336fe9ffccaf9985eaa1fd8ef9220e02b95cd161bc", "kit.part21": "0daf7285daa596a02511eaac2e71a2b152ca88045489c4f06ca756aa7609cd27", "kit.part22": "b599144033d80fe40166edf73ce18051e8981a0f0003574e245038ea0c8fe530", "kit.part23": "88cba0e3428542e2ae03b3af51e3264eab7bda1256cdba2a38d005f2c9813700", "kit.part24": "98dcc6804b02b651fa48ba3952c947fce5c5d66c7b371490846038ef1db13f23", "kit.part25": "4899079ecf014c4ddee5fb39b431cc194900bcbb4ff4b6c88f62f1ed30464c2f", "kit.part26": "7357de21f7bd05056729998d66b15176aeb617f8657698c6ae6f7c024bff38ca", "kit.part27": "b5bfc4e48567c886e436a32575054cde10d74b48d837246bd951ac9abd7539e7", "kit.part28": "fae31d5fd1bfcfea190718bf4c6c8e6f8e4b33c69082a6fedd68af841bf976ee", "kit.part29": "8f0dc3d336e1d9ca2e5f54d71b9817cc2a1786abb090a83e248152f558cacbcf", "kit.part30": "e6e4d9f0b05013492144dcea75572f038d8c46a00981795641e71c6ee3c36780", "kit.part31": "5eacabb58c6710f00976ee8c8227a617756ad82ab5c149824bc5d655aa741ffa", "kit.part32": "ad30f66b4eb60e55b68621491a64327600d4fc8881557f919c1b98bf10f6c6a7", "kit.part33": "ab8249e254fd02de6d07fb6d6c4a58a5dd25dc0401351173b5f6938648c4349c", "kit.part34": "926cbc6cd28d289f2804d1f930246cf0afa8bd4b08ddd0d1b42a58a225e8af8d", "kit.part35": "b87383ddacbd4b82bb759841455c3c55efdffe09e9f709a68515ceb28e314fb8", "kit.part36": "b85877fc99e6e77f637ca45f6f2bcd0c4335468a421ae31d399e585c6d0b3954", "kit.part37": "4764efadcb3752e3c70ebf5d6abcdfb3df67fb873fc5b37dedd22358541e6e7e"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    names = [os.path.basename(p) for p in parts]
    assert names == sorted(PART_SHA), 'STOP: kit_parts 的檔名不對（有 %d 份，要 %d 份；注意有沒有「 (1)」）' % (len(parts), len(PART_SHA))
    tmp = '/content/C10_p44_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                b = f.read()
            assert hashlib.sha256(b).hexdigest() == PART_SHA[os.path.basename(p)], 'STOP: 分割檔雜湊不對 ' + p
            o.write(b)
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])

In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK = '/content/drive/MyDrive/C10/p44', '/content/c10'
KIT_SHA, FILES = 'ff20262c3a4a2ce3e50293958b49f8fe3dcb5567d01ea38f3098efd5230885da', {"data/prereg44/cache/hold/cache.json": "50b75e13511c4333bdf16ebf29ada0a66b21334848f26b562f216838e1498c82", "data/prereg44/cache/hold/codes.npy": "bb0f24fafa849a3bca9990fee6c4415f6150fea9a586f67759eb77570e486f9c", "data/prereg44/cache/hold/meta.npz": "9bc8637a255e9779894a587cec0c0b03558acd913f34fc471eb1e2180107830a", "data/prereg44/cache/hold/off.npy": "72c788c1c978fb79abe446d1f82a5fc4236b715115eb8c7e1db136ed6c7d4984", "data/prereg44/cache/inner/cache.json": "0cba826c04657b361a5cd3d2a8055e844887d412ce40d7f4f305b334db780171", "data/prereg44/cache/inner/codes.npy": "288148d0c936b281aabbc8c839cf214eae5fcf9f03105fc6dea998cfb49d69cd", "data/prereg44/cache/inner/meta.npz": "b1ffa4e68e5f23652de1a7a7b7547cc53ee03fb665b276a31337adc8b2b9e1d7", "data/prereg44/cache/inner/off.npy": "d263aaac0ede2a6459506f3425022af1e67b487cbb2598cb6ed000883642519c", "data/prereg44/cache/train/cache.json": "dd4aa40ee0001fc037e298e470dfb9c15a3df13664b3a5743448dac40053026f", "data/prereg44/cache/train/codes.npy": "79c649e6a3b4e9b5f80bef3ecdccc769830ce9e794891c13b2b1dd49af1f77ba", "data/prereg44/cache/train/meta.npz": "342e920c1130d888de9ffaac4e78249e9a8aa9b651b51aba8a624d226d665990", "data/prereg44/cache/train/off.npy": "ac361d42e4a180de7b98d1fa550a9309923e461df751cdc34e41ccab904dfbfa", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg44/explore/p44_infer.py": "f01ab81811e7da2fb89ba29ef75eea7fb9f2457752571a0e399207dda5620dd3", "results/prereg44/explore/p44_lib.py": "92a2893e9b84a3bd3882e67773286b777dba4e6194c83945c0b07a392d9e213a", "results/prereg44/explore/p44_train.py": "29aeb34928e5e1377d08d379359da2258a1dcbecadaa12800f43f3730dd21480", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.backends.cudnn.version(), torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p44_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p44_kit.zip')
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
RUN = os.path.join(DRIVE, 'run')
os.makedirs(RUN, exist_ok=True)

In [ ]:
# 2. 訓練（不用改；斷線後重按「全部執行」會從最後的存檔接著跑）
rec = os.path.join(RUN, 'train_record.json')
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
done = os.path.exists(rec) and json.load(open(rec)).get('finished')
if not done:
    ck = os.path.join(RUN, 'ckpt')
    if not (os.path.isdir(ck) and any(f.startswith('step_') for f in os.listdir(ck))):
        rc = run(['results/prereg44/explore/p44_train.py', '--workers', workers, '--bench', '300'], os.path.join(RUN, 'bench_log.txt'))
        print('bench exit', rc)
    rc = run(['results/prereg44/explore/p44_train.py', '--workers', workers, '--auto-resume', '--out', RUN, '--record', rec, '--min-free-ram-mb', '2048'], os.path.join(RUN, 'train_log.txt'))
    assert rc == 0, 'STOP: 訓練結束碼 %d（4＝記憶體不足或 STOP 檔，重按「全部執行」會接續）' % rc
print('training finished')

In [ ]:
# 3. 8 個候選版本的每盤 logits：留出題的棋（hold）和 inner 的棋（不用改；已完成的會略過）
LOG = os.path.join(DRIVE, 'logits')
rc = run(['results/prereg44/explore/p44_infer.py', 'candidates', '--ckpt-dir', os.path.join(RUN, 'ckpt'), '--parts', 'hold', 'inner', '--workers', workers, '--out-root', LOG], os.path.join(RUN, 'infer_log.txt'))
assert rc == 0, 'STOP: logits 結束碼 %d' % rc
print('logits finished')

In [ ]:
# 4. 打包回傳（不用改）
res = os.path.join(DRIVE, 'C10_p44_results.zip')
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record.json', 'train_log.txt', 'train_log.jsonl', 'bench_log.txt', 'infer_log.txt'):
        p = os.path.join(RUN, name)
        if os.path.exists(p):
            z.write(p, 'run/' + name)
    for st in (140000, 160000, 180000, 200000):
        z.write(os.path.join(RUN, 'ckpt', 'step_%06d.pt' % st), 'run/ckpt/step_%06d.pt' % st)
    for d in sorted(os.listdir(LOG)):
        for part in ('hold', 'inner'):
            z.write(os.path.join(LOG, d, part + '.npz'), 'logits/%s/%s.npz' % (d, part))
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成。請下載 Drive 的 C10/p44/C10_p44_results.zip 回傳給 Claude（%.0f MB）' % (os.path.getsize(res) / 1e6))